[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/altair-certified/notebooks/day-01-altair-foundations.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · Altair Foundations
**certified-journeys / altair-certified** · Day 1 · Getting Started

> **Goal for today:** Install Altair, create your first charts using marks and encodings, add color and properties, and export a chart to HTML.


In [ ]:
%pip install -q altair vega-datasets


## Step 1 · Import Altair and load a sample dataset

Altair is a declarative visualization library built on the Vega-Lite grammar.
The `vega_datasets` package ships a collection of small, clean datasets perfect
for experimentation.

| Import | Purpose |
|--------|--------|
| `altair` | Chart objects and encoding API |
| `vega_datasets.data` | Access to built-in sample datasets |

A dataset is loaded as a **pandas DataFrame** via `data.<name>()`.


In [ ]:
import altair as alt
from vega_datasets import data

# Load the classic cars dataset (406 rows × 8 columns)
cars = data.cars()

print(cars.shape)
print(cars.dtypes)
cars.head(3)


### What just happened?
- `data.cars()` fetched the dataset from the Vega CDN and returned a pandas DataFrame.
- The DataFrame has columns for `Horsepower`, `Miles_per_Gallon`, `Weight_in_lbs`, `Origin`, and others.
- **All Altair charts are built from DataFrames (or dicts/URLs) — no special data format needed.**
- `dtypes` tells us which columns are numeric vs categorical — important for choosing encoding types.


## Step 2 · Create your first Chart with `mark_point`

Every Altair chart is a `Chart` object with three required ingredients:

1. **Data** — passed to `Chart(data)`
2. **Mark** — `.mark_point()`, `.mark_bar()`, `.mark_line()`, etc.
3. **Encodings** — `.encode(x=..., y=...)` mapping data columns to visual channels

Shorthand syntax for encodings: `'ColumnName:T'` where `T` is one of:

| Type | Code | Example |
|------|------|--------|
| Quantitative | `:Q` | `'Horsepower:Q'` |
| Ordinal | `:O` | `'Cylinders:O'` |
| Nominal | `:N` | `'Origin:N'` |
| Temporal | `:T` | `'Year:T'` |


In [ ]:
# Simplest possible scatter plot: Horsepower vs Miles per Gallon
chart = alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q'
)

chart


### What just happened?
- `alt.Chart(cars)` wraps our DataFrame as the data source.
- **`.mark_point()` declares the chart type** — a scatter plot of individual points.
- `.encode(x=..., y=...)` maps DataFrame columns to visual axes.
- The `:Q` suffix tells Altair these are quantitative (continuous) values, so it uses a linear scale.
- Altair renders the chart inline in Jupyter/Colab automatically.


## Step 3 · Add a color encoding to distinguish categories

Encodings are not limited to `x` and `y`. Altair supports many **visual channels**:

| Channel | Use for |
|---------|---------|
| `color` | Categorical grouping or continuous gradient |
| `size` | Quantitative emphasis |
| `shape` | Categorical differentiation |
| `opacity` | Emphasis or de-emphasis |
| `tooltip` | On-hover labels |

Adding `color='Origin:N'` will automatically assign distinct colors to each origin region
and generate a legend — no extra code needed.


In [ ]:
# Add color encoding for the Origin column (Nominal type)
chart_color = alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin:N',       # Nominal: discrete color per category
    tooltip=['Name', 'Horsepower', 'Miles_per_Gallon', 'Origin']  # hover labels
)

chart_color


### What just happened?
- `color='Origin:N'` added a **legend automatically** — Altair infers that Nominal data needs
  a categorical (non-ordered) color palette.
- `tooltip=[...]` lists column names to show on hover — no JavaScript required.
- **The grammar is composable**: each new encoding channel is just another keyword argument.
- Notice USA cars tend to have higher horsepower and lower fuel efficiency — the color makes this pattern visible instantly.


## Step 4 · Set chart title and adjust size with `.properties()`

`.properties()` controls **non-data visual properties** of the chart:

| Property | Default | Description |
|----------|---------|------------|
| `title` | (none) | Chart title string |
| `width` | 300 | Width in pixels |
| `height` | 300 | Height in pixels |
| `background` | transparent | Background color |

Properties do **not** affect data encodings — they are purely presentational.


In [ ]:
# Add a title and resize the chart
chart_titled = alt.Chart(cars).mark_point().encode(
    x=alt.X('Horsepower:Q', title='Horsepower (hp)'),
    y=alt.Y('Miles_per_Gallon:Q', title='Fuel Efficiency (mpg)'),
    color=alt.Color('Origin:N', title='Country of Origin'),
    tooltip=['Name', 'Horsepower', 'Miles_per_Gallon', 'Origin']
).properties(
    title='Horsepower vs Fuel Efficiency by Origin',
    width=500,
    height=350
)

chart_titled


### What just happened?
- `alt.X(...)` and `alt.Y(...)` are **long-form encodings** that let you set axis titles and
  other per-channel options (scale, sort, bin, etc.).
- `alt.Color(...)` similarly lets you set the legend title.
- **Short-form** (`'Column:Q'`) is fine for quick exploration; **long-form** (`alt.X(...)`) is
  used when you need to customize the channel's appearance.
- `.properties()` is always chained after `.encode()` — order matters in Altair's fluent API.


## Step 5 · Explore another mark: `mark_bar`

The same grammar works for any chart type — just swap the mark.
Here we'll use `mark_bar` to count cars by origin, using Altair's
**built-in aggregation** via `aggregate='count'`.

Altair can aggregate data inside the spec — no need to group by in pandas first:

```
alt.X('Origin:N')         # categorical axis
alt.Y('count()')          # shorthand for COUNT(*)
```


In [ ]:
# Bar chart: count of cars by origin region
bar_chart = alt.Chart(cars).mark_bar().encode(
    x=alt.X('Origin:N', title='Country of Origin'),
    y=alt.Y('count()', title='Number of Models'),
    color='Origin:N',
    tooltip=['Origin:N', 'count()']
).properties(
    title='Number of Car Models by Country of Origin',
    width=300,
    height=250
)

bar_chart


### What just happened?
- `mark_bar()` switched from points to bars — **the encoding logic stayed the same**.
- `'count()'` is Altair shorthand for aggregating by row count — Vega-Lite handles it server-side.
- This is the core of the **declarative grammar**: describe *what* you want (counts by origin),
  not *how* to compute it.
- The chart automatically sorted bars by label; you can override with `sort='x'`, `sort='-y'`, etc.


## Step 6 · Save a chart to HTML

Altair charts can be saved as:

| Format | Method | Notes |
|--------|--------|-------|
| HTML | `chart.save('file.html')` | Self-contained, opens in browser |
| JSON | `chart.save('file.json')` | Vega-Lite spec only |
| PNG/SVG | `chart.save('file.png')` | Requires `altair_saver` + browser driver |

HTML is the easiest — it bundles the Vega-Lite runtime and your spec into one file.


In [ ]:
# Save the titled scatter plot to an HTML file
chart_titled.save('day01_scatter.html')

# Save the bar chart too
bar_chart.save('day01_bar.html')

print("Charts saved to day01_scatter.html and day01_bar.html")
print("Open either file in your browser to see the interactive chart.")

# In Colab, you can also download the file:
# from google.colab import files
# files.download('day01_scatter.html')


### What just happened?
- `chart.save('name.html')` writes a **self-contained HTML file** with the Vega-Lite runtime
  bundled — no server needed, just open in any browser.
- The HTML file preserves all interactivity: tooltips, zoom, pan.
- **Saving to JSON** (`chart.save('name.json')`) exports the raw Vega-Lite spec — useful for
  embedding in other tools or debugging the spec structure.
- In production, you'd typically embed the JSON spec in a web page and load Vega-Lite from a CDN.


## Step 7 · Quick tour of more marks

Let's see three more marks in one go to build your mental model of the grammar:

- `mark_line` — connects points in order (great for time series)
- `mark_area` — like `mark_line` but filled below
- `mark_tick` — small tick marks (great for distributions)


In [ ]:
import pandas as pd

# Mean MPG per year to demonstrate mark_line
mpg_by_year = cars.dropna(subset=['Miles_per_Gallon', 'Year']).copy()
# Compute mean per year group using pandas
mean_mpg = mpg_by_year.groupby('Year', as_index=False)['Miles_per_Gallon'].mean()

line_chart = alt.Chart(mean_mpg).mark_line(point=True).encode(
    x=alt.X('Year:T', title='Year'),          # Temporal encoding for dates
    y=alt.Y('Miles_per_Gallon:Q', title='Avg MPG'),
    tooltip=['Year:T', alt.Tooltip('Miles_per_Gallon:Q', format='.1f')]
).properties(
    title='Average Fuel Efficiency Over Time',
    width=480,
    height=250
)

line_chart


### What just happened?
- `mark_line(point=True)` draws the line **and** adds a dot at each data point — `point=True` is a mark-level static property.
- `'Year:T'` uses the **Temporal** encoding type — Altair formats the axis as dates automatically.
- `alt.Tooltip('Miles_per_Gallon:Q', format='.1f')` formats the tooltip value to 1 decimal place.
- **The trend is clear**: average fuel efficiency improved significantly from 1970 to 1982.


In [ ]:
# Challenge: Build a scatter plot using the 'movies' dataset
# that shows Rotten Tomatoes rating (x) vs IMDB rating (y),
# colored by Major Genre, with a tooltip showing the Title.
# Set the chart title and resize to width=520, height=380.

# Load the movies dataset
movies = data.movies()
print(movies.columns.tolist())

# Your solution here:
# challenge_chart = alt.Chart(movies).mark_point().encode(
#     x=...,
#     y=...,
#     color=...,
#     tooltip=...
# ).properties(
#     title=...,
#     width=...,
#     height=...
# )
# challenge_chart


---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| `alt.Chart(data)` | Wraps your data source; accepts DataFrame, dict, or URL |
| `.mark_*()` | Declares chart type: point, bar, line, area, rect, arc, etc. |
| `.encode(x=, y=, color=, ...)` | Maps data columns to visual channels |
| Shorthand `:Q :N :O :T` | Type hints for quantitative, nominal, ordinal, temporal |
| `alt.X(...)` / long-form | Use when you need axis titles, scales, or sorting |
| `.properties(title=, width=, height=)` | Non-data visual properties |
| `chart.save('file.html')` | Export self-contained interactive chart |
| `count()` aggregation | Built-in row count — no pandas groupby needed |

> **Tip:** The grammar is declarative — you describe what you want, not how to draw it. Think of `mark_*` as chart type and `encode()` as the mapping from data columns to visual channels.

---
## What's next
**Day 2** → Data Shapes & Sources — learn why Altair loves tidy (long-form) data, how to reshape with `melt`, use the `fold` transform, and load data directly from URLs.

Mark Day 1 complete in your [tracker](../index.html).
